In [1]:
import sys
sys.path.append('../src')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Créer les dossiers nécessaires
Path('../reports/figures').mkdir(parents=True, exist_ok=True)
Path('../models').mkdir(parents=True, exist_ok=True)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    classification_report, confusion_matrix, make_scorer
)
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers

from evaluation import (
    classification_metrics, compare_classifiers,
    plot_confusion_matrix, error_analysis_classification,
)

# Désactiver les warnings
import warnings
warnings.filterwarnings('ignore')

print(" Tous les imports sont chargés")

✅ Tous les imports sont chargés


In [2]:
# Charger les données prétraitées
data = joblib.load('../models/processed_data.joblib')
X_train, X_test   = data['X_train'], data['X_test']
y_train, y_test   = data['y_train'], data['y_test']
feature_names     = data['feature_names']
X_test_raw        = data['X_test_raw']

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Churn rate train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')
print(f'Nombre de features: {len(feature_names)}')

# Calcul du scale_pos_weight pour XGBoost
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pw:.2f}')

X_train: (8000, 50) | X_test: (2000, 50)
Churn rate train: 0.102 | test: 0.102
Nombre de features: 50
scale_pos_weight: 8.79


In [3]:
print("="*50)
print("1. RÉGRESSION LOGISTIQUE (Baseline)")
print("="*50)

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

lr_metrics = classification_metrics(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression')
print(lr_metrics)

plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression', '../reports/figures/03_cm_lr.png')

1. RÉGRESSION LOGISTIQUE (Baseline)
{'model': 'Logistic Regression', 'accuracy': 0.697, 'precision': 0.20266272189349113, 'recall': 0.6715686274509803, 'f1': 0.31136363636363634, 'roc_auc': 0.7478983798419145, 'pr_auc': 0.2707798397720261}


<Figure size 500x400 with 1 Axes>

In [4]:
print("="*50)
print("2. RANDOM FOREST")
print("="*50)

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

rf_metrics = classification_metrics(y_test, y_pred_rf, y_proba_rf, 'Random Forest')
print(rf_metrics)

# Feature importance
fi = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
fi.head(12).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest — Feature Importance (top 12)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/03_rf_feature_importance.png', bbox_inches='tight')
plt.show()

2. RANDOM FOREST
{'model': 'Random Forest', 'accuracy': 0.887, 'precision': 0.35526315789473684, 'recall': 0.1323529411764706, 'f1': 0.19285714285714287, 'roc_auc': 0.8022989541027993, 'pr_auc': 0.27121740316034304}


In [5]:
print("="*50)
print("3. XGBOOST (Version de base)")
print("="*50)

xgb_base = xgb.XGBClassifier(
    n_estimators=200,
    scale_pos_weight=scale_pw,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_base.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_xgb = xgb_base.predict(X_test)
y_proba_xgb = xgb_base.predict_proba(X_test)[:, 1]

xgb_metrics = classification_metrics(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost')
print(xgb_metrics)

3. XGBOOST (Version de base)
{'model': 'XGBoost', 'accuracy': 0.872, 'precision': 0.25471698113207547, 'recall': 0.1323529411764706, 'f1': 0.17419354838709677, 'roc_auc': 0.7409807742696188, 'pr_auc': 0.21859080122081936}


In [6]:
print("="*50)
print("3.5 XGBOOST OPTIMISÉ (GridSearch)")
print("="*50)

f1_scorer = make_scorer(f1_score, zero_division=0)

param_grid_xgb = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        n_estimators=200,
        scale_pos_weight=scale_pw,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ),
    param_grid_xgb,
    scoring=f1_scorer,
    cv=3,
    n_jobs=-1,
    verbose=0
)

print("Recherche des meilleurs hyperparamètres...")
xgb_grid.fit(X_train, y_train)

print(f"Meilleurs paramètres: {xgb_grid.best_params_}")
print(f" Melleur F1 (CV): {xgb_grid.best_score_:.4f}")

xgb_optim = xgb_grid.best_estimator_
y_pred_xgb_opt = xgb_optim.predict(X_test)
y_proba_xgb_opt = xgb_optim.predict_proba(X_test)[:, 1]

xgb_opt_metrics = classification_metrics(y_test, y_pred_xgb_opt, y_proba_xgb_opt, 'XGBoost Optimisé')
print(xgb_opt_metrics)

3.5 XGBOOST OPTIMISÉ (GridSearch)
Recherche des meilleurs hyperparamètres...
Meilleurs paramètres: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 1.0}
 Melleur F1 (CV): 0.3842
{'model': 'XGBoost Optimisé', 'accuracy': 0.7155, 'precision': 0.24186704384724186, 'recall': 0.8382352941176471, 'f1': 0.3754116355653128, 'roc_auc': 0.8120005240403511, 'pr_auc': 0.32691641991688974}


In [7]:
print("="*50)
print("4.5 MLP AMÉLIORÉ (Architecture plus profonde)")
print("="*50)

def build_improved_mlp(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        
        keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.4),
        
        keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.1),
        
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    optimizer = keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])
    return model

mlp_improved = build_improved_mlp(X_train.shape[1])

history_improved = mlp_improved.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    ],
    class_weight={0: 1.0, 1: scale_pw * 1.2},
    verbose=0
)

print(f"Epochs effectuées: {len(history_improved.history['loss'])}")

y_proba_mlp_imp = mlp_improved.predict(X_test, verbose=0).flatten()
y_pred_mlp_imp = (y_proba_mlp_imp >= 0.45).astype(int)

mlp_imp_metrics = classification_metrics(y_test, y_pred_mlp_imp, y_proba_mlp_imp, 'MLP Amélioré')
print(mlp_imp_metrics)

# Courbes d'apprentissage
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_improved.history['loss'], label='Train Loss')
ax.plot(history_improved.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Crossentropy')
ax.set_title('MLP Amélioré — Courbes d\'apprentissage')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/03_mlp_improved_curves.png', bbox_inches='tight')
plt.show()

4.5 MLP AMÉLIORÉ (Architecture plus profonde)
Epochs effectuées: 73
{'model': 'MLP Amélioré', 'accuracy': 0.761, 'precision': 0.23954372623574144, 'recall': 0.6176470588235294, 'f1': 0.3452054794520548, 'roc_auc': 0.7441755098475917, 'pr_auc': 0.24013999708124711}


In [8]:
print("="*50)
print("5. STACKING (Combinaison des modèles)")
print("="*50)

base_models = [
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )),
    ('xgb', xgb.XGBClassifier(
        n_estimators=200,
        scale_pos_weight=scale_pw,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ))
]

stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(class_weight='balanced'),
    cv=5,
    n_jobs=-1
)

print("Entraînement du stacking...")
stacking.fit(X_train, y_train)

y_pred_stack = stacking.predict(X_test)
y_proba_stack = stacking.predict_proba(X_test)[:, 1]

stack_metrics = classification_metrics(y_test, y_pred_stack, y_proba_stack, 'Stacking')
print(stack_metrics)

5. STACKING (Combinaison des modèles)
Entraînement du stacking...
{'model': 'Stacking', 'accuracy': 0.724, 'precision': 0.24782608695652175, 'recall': 0.8382352941176471, 'f1': 0.3825503355704698, 'roc_auc': 0.8117166688501682, 'pr_auc': 0.3007701226412408}


In [10]:
print("="*50)
print("7. COMPARAISON COMPLÈTE DES MODÈLES")
print("="*50)

# Rassembler toutes les métriques
all_metrics = [
    lr_metrics,
    rf_metrics,
    xgb_metrics,
    xgb_opt_metrics,
    mlp_imp_metrics,
    stack_metrics,
]

# Ajouter SMOTE si disponible
if 'xgb_smote_metrics' in locals() and xgb_smote_metrics is not None:
    all_metrics.append(xgb_smote_metrics)

# Créer le tableau comparatif
comparison = compare_classifiers(all_metrics)
print("=== Tableau comparatif complet ===")
display(comparison)

# Sauvegarder
joblib.dump(comparison, '../models/churn_comparison_complete.joblib')

# Visualisation
import plotly.graph_objects as go

metrics_radar = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
metrics_radar = [m for m in metrics_radar if m in comparison.columns]

fig = go.Figure()
for model_name in comparison.index[:5]:  # Top 5
    values = comparison.loc[model_name, metrics_radar].tolist()
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=metrics_radar + [metrics_radar[0]],
        fill='toself',
        name=model_name,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1])),
    title='Comparaison radar — Top 5 modèles'
)
fig.write_html('../reports/figures/03_radar_comparison_complete.html')
fig.show()

7. COMPARAISON COMPLÈTE DES MODÈLES
=== Tableau comparatif complet ===


,accuracy,precision,recall,f1,roc_auc,pr_auc
model,,,,,,
Stacking,0.7240,0.2478,0.8382,0.3826,0.8117,0.3008
XGBoost Optimisé,0.7155,0.2419,0.8382,0.3754,0.8120,0.3269
MLP Amélioré,0.7610,0.2395,0.6176,0.3452,0.7442,0.2401
Logistic Regression,0.6970,0.2027,0.6716,0.3114,0.7479,0.2708
Random Forest,0.8870,0.3553,0.1324,0.1929,0.8023,0.2712
XGBoost,0.8720,0.2547,0.1324,0.1742,0.7410,0.2186


In [11]:
print("="*50)
print("8. COURBES ROC ET PR")
print("="*50)

models_proba = {
    'Logistic Regression': y_proba_lr,
    'Random Forest': y_proba_rf,
    'XGBoost Optimisé': y_proba_xgb_opt,
    'MLP Amélioré': y_proba_mlp_imp,
    'Stacking': y_proba_stack,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for name, proba in models_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=name)
axes[0].plot([0,1],[0,1],'k--', alpha=0.5)
axes[0].set_title('Courbes ROC')
axes[0].set_xlabel('Taux de faux positifs')
axes[0].set_ylabel('Taux de vrais positifs')
axes[0].legend()

# PR Curves
for name, proba in models_proba.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, label=name)
axes[1].set_title('Courbes Precision-Recall')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/03_roc_pr_curves_complete.png', bbox_inches='tight')
plt.show()

8. COURBES ROC ET PR


In [12]:
print("="*50)
print("9. RECHERCHE DU MEILLEUR SEUIL")
print("="*50)

thresholds = np.arange(0.20, 0.75, 0.05)
results_threshold = {}

for name, proba in models_proba.items():
    best_f1 = 0
    best_t = 0.5
    best_recall = 0
    
    for t in thresholds:
        y_pred_t = (proba >= t).astype(int)
        f1_t = f1_score(y_test, y_pred_t, zero_division=0)
        if f1_t > best_f1:
            best_f1 = f1_t
            best_t = t
            best_recall = recall_score(y_test, y_pred_t)
    
    results_threshold[name] = {
        'seuil_optimal': best_t,
        'f1_optimal': best_f1,
        'recall_optimal': best_recall
    }
    print(f"{name}: seuil={best_t:.2f}, F1={best_f1:.4f}, Recall={best_recall:.4f}")

# Visualisation du meilleur modèle
best_model_name = comparison.loc[comparison['f1'].idxmax()].name
best_proba = models_proba[best_model_name]

f1_scores = [f1_score(y_test, (best_proba >= t).astype(int), zero_division=0) for t in thresholds]
rec_scores = [recall_score(y_test, (best_proba >= t).astype(int), zero_division=0) for t in thresholds]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores, label='F1', marker='o')
ax.plot(thresholds, rec_scores, label='Recall', marker='s', linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', label='Seuil par défaut (0.5)')
ax.axvline(results_threshold[best_model_name]['seuil_optimal'], 
           color='red', linestyle='--', label=f"Seuil optimal: {results_threshold[best_model_name]['seuil_optimal']:.2f}")
ax.set_xlabel('Seuil de décision')
ax.set_title(f'F1 et Recall selon le seuil — {best_model_name}')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/03_threshold_analysis_complete.png', bbox_inches='tight')
plt.show()

9. RECHERCHE DU MEILLEUR SEUIL
Logistic Regression: seuil=0.70, F1=0.3620, Recall=0.3922
Random Forest: seuil=0.30, F1=0.3974, Recall=0.7549
XGBoost Optimisé: seuil=0.65, F1=0.3847, Recall=0.5931
MLP Amélioré: seuil=0.45, F1=0.3452, Recall=0.6176
Stacking: seuil=0.55, F1=0.3827, Recall=0.8039


In [16]:
print("="*50)
print("10. MEILLEUR MODÈLE ET SAUVEGARDE")
print("="*50)

# Trouver le meilleur modèle
best_name = comparison.loc[comparison['f1'].idxmax()].name
print(f"MEILLEUR MODÈLE: {best_name}")
print(f"Métriques: {comparison.loc[best_name].to_dict()}")

# Associer le nom au modèle
all_models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb_base,
    'XGBoost Optimisé': xgb_optim,
    'MLP Amélioré': mlp_improved,
    'Stacking': stacking,
}
if 'xgb_smote' in locals():
    all_models['XGB + SMOTE'] = xgb_smote

best_model = all_models[best_name]
best_proba_final = models_proba[best_name]
best_threshold = results_threshold[best_name]['seuil_optimal']
best_pred_final = (best_proba_final >= best_threshold).astype(int)

# Métriques avec seuil optimisé
f1_opt = f1_score(y_test, best_pred_final)
recall_opt = recall_score(y_test, best_pred_final)
prec_opt = precision_score(y_test, best_pred_final)

print(f"\nAvec seuil optimisé ({best_threshold:.2f}):")
print(f"   Precision: {prec_opt:.4f}")
print(f"   Recall: {recall_opt:.4f}")
print(f"   F1: {f1_opt:.4f}")

# Sauvegarde
if 'MLP' in best_name:
    best_model.save('../models/best_model_churn_final.keras')
else:
    joblib.dump(best_model, '../models/best_model_churn_final.joblib')

# Métadonnées
joblib.dump({
    'model_name': best_name,
    'metrics': comparison.loc[best_name].to_dict(),
    'threshold': best_threshold,
    'threshold_metrics': {
        'precision': prec_opt,
        'recall': recall_opt,
        'f1': f1_opt
    }
}, '../models/best_model_metadata_final.joblib')

# Probabilités
pd.DataFrame({
    'churn_proba': best_proba_final,
    'churn_pred': best_pred_final,
    'total_revenue': X_test_raw['total_revenue'].values if 'total_revenue' in X_test_raw.columns else 0,
}).to_parquet('../models/churn_proba_test_final.parquet', index=False)

print(f"\n Modèle sauvegardé dans: models/best_model_churn_final")
print("→ Prochaine étape: 04_modeling_revenue_risk.ipynb")

10. MEILLEUR MODÈLE ET SAUVEGARDE
MEILLEUR MODÈLE: Stacking
Métriques: {'accuracy': 0.724, 'precision': 0.2478, 'recall': 0.8382, 'f1': 0.3826, 'roc_auc': 0.8117, 'pr_auc': 0.3008}

Avec seuil optimisé (0.55):
   Precision: 0.2511
   Recall: 0.8039
   F1: 0.3827

 Modèle sauvegardé dans: models/best_model_churn_final
→ Prochaine étape: 04_modeling_revenue_risk.ipynb
